In [3]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [5]:
import gensim

In [57]:
from gensim.models import Word2Vec

model_CBOW = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

model_S = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

In [66]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_CBOW.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['despoiled', 'initial', 'whereas', 'twisting', 'ache', '102', 'offends', 'fobic', 'mistakenly', 'grants', 'Unfair', 'protestation', 'devastated', 'weakness', 'bunked', 'racially', 'secrets', 'tile', 'Coping', 'Cadet', 'blankness', 'immature', 'Showjumping', 'rioting', 'student-house', 'resullt', 'lard', 'bouncer', 'curtain', 'locally']


In [67]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_S.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['despoiled', 'initial', 'whereas', 'twisting', 'ache', '102', 'offends', 'fobic', 'mistakenly', 'grants', 'Unfair', 'protestation', 'devastated', 'weakness', 'bunked', 'racially', 'secrets', 'tile', 'Coping', 'Cadet', 'blankness', 'immature', 'Showjumping', 'rioting', 'student-house', 'resullt', 'lard', 'bouncer', 'curtain', 'locally']


In [60]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_CBOW.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage CBOW: {token_coverage:.2f}%")

Token-level coverage CBOW: 97.18%


In [61]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_S.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage S: {token_coverage:.2f}%")

Token-level coverage S: 97.18%


In [65]:
print(model_CBOW.wv.most_similar("leukemia", topn=10))
print(model_S.wv.most_similar("leukemia", topn=10))

[('cancer', 0.9516650438308716), ('stolen', 0.9378918409347534), ('seriously', 0.9281783699989319), ('beloved', 0.9250959753990173), ('brother-in-law', 0.9234688878059387), ('personality', 0.9233343005180359), ('80', 0.9207361340522766), ('stealing', 0.9207311272621155), ('beaten', 0.917194664478302), ('Swedish', 0.9158850908279419)]
[('paternal', 0.9695711135864258), ('gradmother', 0.9648404717445374), ('thirteen', 0.962873637676239), ('lung', 0.9575186371803284), ('tragically', 0.9532632231712341), ('eleven', 0.9412897825241089), ('cerebral', 0.9410162568092346), ('collapsed', 0.9397276043891907), ('reared', 0.9312583208084106), ('heart-attack', 0.9289187788963318)]


In [63]:
print([word for word in model_vocab if len(word) <= 2])
print([word for word in dataset_vocab if len(word) <= 2])

['$', '50', '9', 'An', '11', '5', '35', '30', 'My', 'we', 'In', '80', '20', 'Is', ')', 'it', 'us', ':', 'ok', '(', '2A', '84', '15', 'at', 'km', 'no', '40', 'as', 'D', 'II', '21', 'in', 'so', 'TV', 'NO', 'Of', 'It', 'ex', 'OK', '60', 'to', '3', '-', 'C', '18', '17', 'He', '6', 'At', 'me', 'do', 'am', '2', 'go', '22', 'e', '10', 'C+', '?', 'an', 'On', 'A', '12', 'Mr', '14', 'As', "'d", '16', 'I', "'", 'my', '``', '/', "''", 'I.', 'he', 'Oh', '&', '!', 'a', 'We', ';', '8', 'LP', ']', 'wo', 'B+', '[', '.', 'be', '1', 'B', "'m", 'pm', 'if', 'Ca', 'TA', 'm', 'To', 'F4', 'of', '19', '13', 'aa', 'Ye', '4', 'PY', ',', 'So', "'s", '7', 'X', 'i', 'TE', 'Km', 'If', 'ca', 'No', 'or', 'up', 'on', 'is', 'by', '%', 'By']
['$', '50', '9', 'An', '11', '5', '35', '26', '30', 'My', 'we', '79', 'In', '80', '20', 'Is', '--', ')', 'it', 'CV', 'us', '2C', ':', 'ok', '(', '2A', '84', '15', 'Mt', 'at', 'km', 'no', '40', 'as', 'D', 'II', '21', 'g', 'in', 'S.', 'so', 'TV', 'NO', '+2', '5B', 'Of', 'ma', 'It', 'ex